# ch05 Bonus 07：内存高效加载（Memory-Efficient Weight Loading）

> 对照官方 `ch05/08_memory_efficient_weight_loading`

## 一句话

加载大模型权重时，普通方法会占用 2-3 倍内存（原始权重 + 模型副本 + 中间张量）。本 notebook 演示 **逐层加载 + meta device + 低精度** 来压低峰值内存。

## 问题：普通加载的内存放大

```
# 普通做法：先创建模型（占内存 A），再把权重 load 进去（又占一份临时内存 B）
model = GPTModel(cfg)          # 分配内存 A
model.load_state_dict(torch.load("weights.pt"))  # 临时内存 B（≈A）
```
峰值内存 ≈ 2-3 倍模型大小。7B 模型 fp32 要 ~84GB 内存才能加载！

## 三种省内存手段

| 手段 | 效果 |
|------|------|
| **meta device 初始化** | 先在 meta 设备创建模型（不占内存），再逐层填充真实权重 |
| **逐层加载** | 一次只加载一层到内存，用完释放 |
| **低精度 (fp16/bf16)** | 直接减半内存 |

In [ ]:
import torch
from src.gpt import GPTModel, GPT_CONFIG_124M

# 方法 1：meta device —— 创建模型但不分配真实内存
print("方法 1：meta device")
cfg = dict(GPT_CONFIG_124M)
with torch.device("meta"):
    meta_model = GPTModel(cfg)
# meta 模型的参数不占真实内存，是占位符
p0 = next(meta_model.parameters())
print(f"  meta 模型参数 device: {p0.device}")
print(f"  （真实内存占用 ≈ 0，只是元信息）")

# 方法 2：对比普通加载的内存占用
import os
torch.manual_seed(123)
normal_model = GPTModel(cfg)
n_params = sum(p.numel() for p in normal_model.parameters())
mem_bytes = n_params * 4  # fp32 每个 4 字节
print(f"\n普通模型参数量: {n_params:,} = {mem_bytes/1e9:.2f} GB (fp32)")
print(f"普通 load_state_dict 峰值内存 ≈ 2× = {2*mem_bytes/1e9:.2f} GB")
print(f"用 meta + 逐层加载可降到 ≈ 1× = {mem_bytes/1e9:.2f} GB")

In [ ]:
# 方法 3：演示「逐参数赋值」的低内存加载思路
# 真实场景：把权重存成多个小文件（每层一个），逐个加载并赋值给 meta 模型

def materialize_from_meta(meta_model, materialize_fn):
    """把 meta 模型逐参数转成真实张量（模拟逐层加载）。"""
    # to_empty 把 meta 参数变成真实内存，但值未初始化
    real_model = meta_model.to_empty(device="cpu")
    # 逐参数用 materialize_fn 填充（真实场景是从磁盘逐层读权重）
    with torch.no_grad():
        for name, p in real_model.named_parameters():
            # 这里用随机值演示；真实场景是 p.copy_(loaded_weights[name])
            materialize_fn(p)
    return real_model

torch.manual_seed(0)
real = materialize_from_meta(meta_model, lambda p: p.normal_())
print(f"meta 模型已实例化为真实模型，参数已填充")
print(f"参数现在占真实内存: {sum(p.numel() for p in real.parameters())*4/1e9:.2f} GB")

# 验证能正常前向
import tiktoken
tok = tiktoken.get_encoding("gpt2")
idx = torch.tensor([tok.encode("Hello")])
real.eval()
with torch.no_grad():
    out = real(idx)
print(f"前向输出形状: {tuple(out.shape)} ✓")